# Ames Housing: Data Cleaning

Part 1 of the Ames Housing project. This notebook investigates and cleans missing values in the raw `train.csv` dataset, producing `train_cleaned.csv` for the analysis notebook that follows (`02_eda.ipynb`).

**Initial expectations, before touching the data:** no strong hypothesis yet about the exact hierarchy of features driving `SalePrice`, but general intuition suggests land area, location, and room count are likely among the strongest predictors. Untested for now, revisited in the analysis notebook.

**Approach:** identify columns with missing values or low information value, and decide, case by case, whether to fill or drop them. No column is dropped or filled without a specific reason backed by a check against the data, not assumption alone.

In [1]:
import pandas as pd

df = pd.read_csv("../data/train.csv")
df["MSSubClass"] = df["MSSubClass"].astype(str)  # stored as a number but is a category code, not a quantity

missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

numeric_cols = df.select_dtypes(include="number").columns.tolist()
object_cols = df.select_dtypes(include=["object", "str"]).columns.tolist()

missing

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtExposure      38
BsmtFinType2      38
BsmtQual          37
BsmtCond          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
dtype: int64

## Splitting Missing Columns

Dropping `Id` first, since it is a row index with no analytical value. Then splitting the remaining missing columns into two groups: significant (more than 10% missing) and insignificant (10% or less), each further split by data type.

In [2]:
df.drop("Id", axis=1, inplace=True)

sig_missing = missing[missing > (0.1 * len(df))].sort_values(ascending=False)
sig_numeric = sig_missing.index.intersection(numeric_cols)
sig_categorical = sig_missing.index.intersection(object_cols)

insig_missing = missing[missing <= (0.1 * len(df))].sort_values(ascending=False)
insig_numeric = insig_missing.index.intersection(numeric_cols)
insig_categorical = insig_missing.index.intersection(object_cols)

print("Significant:")
print(sig_missing)
print("\nInsignificant:")
print(insig_missing)

Significant:
PoolQC         1453
MiscFeature    1406
Alley          1369
Fence          1179
MasVnrType      872
FireplaceQu     690
LotFrontage     259
dtype: int64

Insignificant:
GarageType      81
GarageYrBlt     81
GarageFinish    81
GarageQual      81
GarageCond      81
BsmtExposure    38
BsmtFinType2    38
BsmtQual        37
BsmtCond        37
BsmtFinType1    37
MasVnrArea       8
Electrical       1
dtype: int64


## Garage Columns

All five garage-related columns share the same missing count: 81. Checking `GarageArea`, which has no missing values, to confirm this means no garage rather than a genuine gap.

In [3]:
no_garage = df[df["GarageQual"].isnull()][["GarageType", "GarageYrBlt", "GarageFinish", "GarageQual", "GarageCond", "GarageArea"]]
no_garage["GarageArea"].unique()

array([0])

Confirmed: `GarageArea` is 0 for all 81 rows. These are houses with no garage, not missing data. Filling with the mean or mode would invent a garage that does not exist, so the categorical columns get `"None"` and `GarageYrBlt` gets 0.

In [4]:
garage_cols_numeric = ["GarageYrBlt"]
garage_cols_categorical = ["GarageType", "GarageFinish", "GarageQual", "GarageCond"]

df[garage_cols_numeric] = df[garage_cols_numeric].fillna(0)
df[garage_cols_categorical] = df[garage_cols_categorical].fillna("None")

## Basement Columns

Five basement columns are missing, but not by the same amount: `BsmtExposure` and `BsmtFinType2` are missing 38, while `BsmtQual`, `BsmtCond`, and `BsmtFinType1` are missing 37. A 1-row difference worth investigating before assuming this is the same clean pattern as Garage.

First, checking that the 37s are on the same rows.

In [5]:
qual_missing = set(df[df["BsmtQual"].isnull()].index)
cond_missing = set(df[df["BsmtCond"].isnull()].index)
exposure_missing = set(df[df["BsmtExposure"].isnull()].index)
fintype1_missing = set(df[df["BsmtFinType1"].isnull()].index)
fintype2_missing = set(df[df["BsmtFinType2"].isnull()].index)

qual_missing == cond_missing == fintype1_missing

True

True. Next, checking that the 38s are supersets of the 37s.

In [6]:
exposure_missing.issuperset(qual_missing) & fintype2_missing.issuperset(qual_missing)

True

True. Now checking whether the two 38s are on the same row as each other.

In [7]:
exposure_missing == fintype2_missing

False

False. Both are supersets of the 37s, but their extra row is different in each case. Finding exactly where they differ.

In [8]:
exposure_missing.symmetric_difference(fintype2_missing)

{332, 948}

In [9]:
df.loc[[332, 948], ["BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2", "TotalBsmtSF"]]

,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinType2,TotalBsmtSF
332,Gd,TA,No,GLQ,NaN,3206
948,Gd,TA,NaN,Unf,Unf,936


Both rows have a real, nonzero `TotalBsmtSF`: these houses genuinely have a basement, and are each missing exactly one field. Not a "no basement" case. Fixing each with the mode of its column.

In [10]:
df.loc[332, "BsmtFinType2"] = df["BsmtFinType2"].mode()[0]
df.loc[948, "BsmtExposure"] = df["BsmtExposure"].mode()[0]

With the two exceptions handled, confirming the remaining missing rows genuinely correspond to houses with no basement (`TotalBsmtSF == 0`).

In [11]:
qual_missing = set(df[df["BsmtQual"].isnull()].index)
cond_missing = set(df[df["BsmtCond"].isnull()].index)
fintype1_missing = set(df[df["BsmtFinType1"].isnull()].index)
no_basement = set(df[df["TotalBsmtSF"] == 0].index)

qual_missing == cond_missing == fintype1_missing == no_basement

True

Confirmed. Checking whether any other basement columns exist that were not part of this missing group.

In [12]:
[col for col in df.columns if "Bsmt" in col]

['BsmtQual',
 'BsmtCond',
 'BsmtExposure',
 'BsmtFinType1',
 'BsmtFinSF1',
 'BsmtFinType2',
 'BsmtFinSF2',
 'BsmtUnfSF',
 'TotalBsmtSF',
 'BsmtFullBath',
 'BsmtHalfBath']

In [13]:
bsmt_categorical = ["BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"]
bsmt_numeric_complete = ["BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF", "BsmtFullBath", "BsmtHalfBath"]

no_basement_rows = df[df[bsmt_categorical].isnull().any(axis=1)]
no_basement_rows[bsmt_numeric_complete].isnull().sum()

BsmtFinSF1      0
BsmtFinSF2      0
BsmtUnfSF       0
TotalBsmtSF     0
BsmtFullBath    0
BsmtHalfBath    0
dtype: int64

All numeric basement columns are complete for these rows, consistent with no basement. Filling the categorical basement columns with `"None"`.

In [14]:
bsmt_cols = ["BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"]
df.fillna({col: "None" for col in bsmt_cols}, inplace=True)

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,60,RL,62.0,7917,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,8,2007,WD,Normal,175000
1456,20,RL,85.0,13175,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,MnPrv,NaN,0,2,2010,WD,Normal,210000
1457,70,RL,66.0,9042,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,GdPrv,Shed,2500,5,2010,WD,Normal,266500
1458,20,RL,68.0,9717,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,142125


## MasVnrType and MasVnrArea

Both masonry veneer columns have missing values, but `MasVnrType` has far more than `MasVnrArea`. Checking whether `MasVnrArea`'s missing rows are a subset of `MasVnrType`'s.

In [15]:
type_missing = set(df[df["MasVnrType"].isnull()].index)
area_missing = set(df[df["MasVnrArea"].isnull()].index)

area_missing.issubset(type_missing)

True

True. Now checking the rows where `MasVnrType` is missing but `MasVnrArea` is not, hoping that every such row has `MasVnrArea` equal to 0, which would confirm the missing type simply means no veneer.

In [16]:
mismatch = df[df["MasVnrType"].isnull() & df["MasVnrArea"].notna()]
mismatch["MasVnrArea"].value_counts()

MasVnrArea
0.0      859
1.0        2
288.0      1
344.0      1
312.0      1
Name: count, dtype: int64

Mostly 0, but 5 rows have a real, nonzero area. Checking those directly.

In [17]:
mismatch.loc[mismatch["MasVnrArea"] > 0, ["MasVnrType", "MasVnrArea"]]

,MasVnrType,MasVnrArea
624,NaN,288.0
773,NaN,1.0
1230,NaN,1.0
1300,NaN,344.0
1334,NaN,312.0


These 5 houses genuinely have a veneer but are missing the type label. Filling with the mode.

In [18]:
df.loc[[624, 773, 1230, 1300, 1334], "MasVnrType"] = df["MasVnrType"].mode()[0]

Filling the remaining missing `MasVnrArea` values with 0, then checking whether `area_zero_rows` and `type_null_rows` now match exactly before filling `MasVnrType` in bulk.

In [19]:
df["MasVnrArea"] = df["MasVnrArea"].fillna(0)

area_zero_rows = set(df[df["MasVnrArea"] == 0].index)
type_null_rows = set(df[df["MasVnrType"].isnull()].index)

area_zero_rows == type_null_rows

False

False. Checking the size of the mismatch, and whether `type_null_rows` is at least fully contained in `area_zero_rows`.

In [20]:
print(len(area_zero_rows), len(type_null_rows))
type_null_rows.issubset(area_zero_rows)

869 867


True

True, so every "no type" row is correctly reflected as "no area." The mismatch runs the other way: some rows have `MasVnrArea == 0` but a real, recorded type. Isolating them.

In [21]:
extra_rows = area_zero_rows - type_null_rows
df.loc[list(extra_rows), ["MasVnrType", "MasVnrArea"]]

,MasVnrType,MasVnrArea
688,BrkFace,0.0
1241,Stone,0.0


Rows 688 and 1241 have a valid type (`BrkFace` and `Stone`) but 0 area, which does not make physical sense for a real veneer.

**Verdict:** treating `MasVnrType` as the more trustworthy field here. Recording a veneer type is a quick visual judgment, while measuring area takes real effort, so a 0 alongside a real type more likely reflects an incomplete measurement than a genuine absence of veneer. Fixing the area instead of wiping the type, using the median area for that same type among houses with a real, nonzero value.

In [22]:
brkface_median = df[(df["MasVnrType"] == "BrkFace") & (df["MasVnrArea"] > 0)]["MasVnrArea"].median()
stone_median = df[(df["MasVnrType"] == "Stone") & (df["MasVnrArea"] > 0)]["MasVnrArea"].median()

df.loc[688, "MasVnrArea"] = brkface_median
df.loc[1241, "MasVnrArea"] = stone_median

area_zero_rows = set(df[df["MasVnrArea"] == 0].index)
type_null_rows = set(df[df["MasVnrType"].isnull()].index)
area_zero_rows == type_null_rows

True

Now they match exactly. Filling the remaining `MasVnrType` nulls with `"None"`.

In [23]:
df["MasVnrType"] = df["MasVnrType"].fillna("None")

## Electrical

Only 1 missing value, and unlike Garage, Basement, or MasVnr, there is no reasonable "does not apply" interpretation: every house has an electrical system. This is a genuine data-entry gap, filled with the mode.

In [24]:
df["Electrical"] = df["Electrical"].fillna(df["Electrical"].mode()[0])

missing_now = df.isnull().sum()
missing_now = missing_now[missing_now > 0].sort_values(ascending=False)
missing_now

PoolQC         1453
MiscFeature    1406
Alley          1369
Fence          1179
FireplaceQu     690
LotFrontage     259
dtype: int64

All insignificant missing values are resolved, with a bonus: `MasVnrType`/`MasVnrArea` from the significant group was fixed along the way. Moving to the remaining significant columns.

## PoolQC

`PoolQC` (pool quality) and `PoolArea` (pool area) are both present in the data. Hypothesis: wherever `PoolQC` is missing, `PoolArea` is 0.

In [25]:
poolQ = set(df[df["PoolQC"].isnull()].index)
poolA = set(df[df["PoolArea"] == 0].index)

poolQ == poolA

True

Confirmed. Every missing `PoolQC` value genuinely means no pool. Filling with `"None"`.

In [26]:
df["PoolQC"] = df["PoolQC"].fillna("None")

## MiscFeature and MiscVal

Same approach as `PoolQC`: checking whether `MiscFeature` missing lines up with `MiscVal` equal to 0.

In [27]:
Misc_feat = set(df[df["MiscFeature"].isnull()].index)
Misc_val = set(df[df["MiscVal"] == 0].index)

Misc_feat == Misc_val

False

False. Finding where they differ.

In [28]:
Misc_feat.symmetric_difference(Misc_val)

{873, 1200}

In [29]:
df.loc[[873, 1200], ["MiscFeature", "MiscVal"]]

,MiscFeature,MiscVal
873,Othr,0
1200,Shed,0


Same situation as `MasVnrType`/`MasVnrArea`: rows 873 and 1200 have a real, recorded feature (`Othr` and `Shed`) but a value of 0. Recording what the feature is takes less effort than appraising its value, so this looks like incomplete data entry rather than a genuinely worthless feature. Fixing with the median value for that feature type.

In [30]:
othr_median = df[(df["MiscFeature"] == "Othr") & (df["MiscVal"] > 0)]["MiscVal"].median()
shed_median = df[(df["MiscFeature"] == "Shed") & (df["MiscVal"] > 0)]["MiscVal"].median()

df.loc[873, "MiscVal"] = othr_median
df.loc[1200, "MiscVal"] = shed_median

Misc_feat = set(df[df["MiscFeature"].isnull()].index)
Misc_val = set(df[df["MiscVal"] == 0].index)
Misc_feat == Misc_val

True

Matches now. Filling the remaining `MiscFeature` nulls with `"None"`.

In [31]:
df["MiscFeature"] = df["MiscFeature"].fillna("None")

## Alley

Per the data description, `NA` is a valid category meaning no alley access, not a genuine gap, consistent with the roughly 94% missing rate. Checking the relationship with `SalePrice` before filling, given how rare alley access is.

In [32]:
df.groupby("Alley")["SalePrice"].mean()

Alley
Grvl    122219.080000
Pave    168000.585366
Name: SalePrice, dtype: float64

A real category, not missing data. No numeric counterpart to cross-check against here, unlike Garage, Pool, or Basement, but the data description directly confirms what `NA` means. Filling with `"None"`.

In [33]:
df["Alley"] = df["Alley"].fillna("None")

## Fence

Same approach as `Alley`: checking the relationship with `SalePrice` before deciding how to treat the missing values.

In [34]:
df.groupby("Fence")["SalePrice"].mean()

Fence
GdPrv    178927.457627
GdWo     140379.314815
MnPrv    148751.089172
MnWw     134286.363636
Name: SalePrice, dtype: float64

The spread across fence types is not dramatic, but keeping the column rather than dropping it. Filling with `"None"` and checking the group averages again, since the no-fence category is by far the largest group.

In [35]:
df["Fence"] = df["Fence"].fillna("None")
df.groupby("Fence")["SalePrice"].mean()

Fence
GdPrv    178927.457627
GdWo     140379.314815
MnPrv    148751.089172
MnWw     134286.363636
None     187596.837998
Name: SalePrice, dtype: float64

Counterintuitive finding: houses with no fence have the highest average `SalePrice` (187,597), higher than every fenced category. Expected the opposite. Testing a possible explanation: fences tend to appear on smaller, more private lots, while larger properties often do not need one.

In [36]:
df.groupby("Fence")["LotArea"].mean()

Fence
GdPrv    10520.288136
GdWo      9634.018519
MnPrv     9204.458599
MnWw      9250.909091
None     10743.659881
Name: LotArea, dtype: float64

No-fence properties do have the largest average `LotArea`, and `GdPrv` is close behind, mildly consistent with the lot-size theory. But the gap (roughly 1,000 to 1,500 sq ft) is too small to fully explain the roughly $50,000 difference in average `SalePrice` between `None` and the cheapest fence category. **Verdict:** lot size is likely part of the explanation, not the whole story. Left open rather than forcing a fuller conclusion than the data supports.

## FireplaceQu

Has a numeric counterpart, `Fireplaces` (count of fireplaces). Same verification as Garage and Pool: if `FireplaceQu` is missing, `Fireplaces` should be exactly 0.

In [37]:
fireqc_missing = set(df[df["FireplaceQu"].isnull()].index)
fireplaces_zero = set(df[df["Fireplaces"] == 0].index)

fireqc_missing == fireplaces_zero

True

Confirmed, no exceptions. Filling with `"None"`.

In [38]:
df["FireplaceQu"] = df["FireplaceQu"].fillna("None")

## LotFrontage

259 missing, about 18%. Unlike Garage, Pool, or Basement, there is no reason to believe a missing value here means "no frontage": every house has a lot with some width facing the street. This is genuine missing data. Before defaulting to a flat mean or median, checking whether a smarter, more informed fill is possible.

In [39]:
df["LotFrontage"].corr(df["SalePrice"])

np.float64(0.3517990965706781)

A real, moderate relationship (0.351), so this column is worth keeping and filling carefully. Checking whether `LotArea` can help predict missing frontage, since both describe the lot.

In [40]:
df["LotFrontage"].corr(df["LotArea"])

np.float64(0.4260950187718078)

Moderate (0.42), not strong enough to rely on alone. Hypothesis: houses in the same `Neighborhood` likely share similar lot layouts, so neighborhood might predict frontage better than lot area does. Testing this before deciding.

In [41]:
df.groupby("Neighborhood")["LotFrontage"].median()

Neighborhood
Blmngtn    43.0
Blueste    24.0
BrDale     21.0
BrkSide    52.0
ClearCr    80.0
CollgCr    70.0
Crawfor    74.0
Edwards    65.5
Gilbert    65.0
IDOTRR     60.0
MeadowV    21.0
Mitchel    73.0
NAmes      73.0
NPkVill    24.0
NWAmes     80.0
NoRidge    91.0
NridgHt    88.5
OldTown    60.0
SWISU      60.0
Sawyer     71.0
SawyerW    66.5
Somerst    73.5
StoneBr    61.5
Timber     85.0
Veenker    68.0
Name: LotFrontage, dtype: float64

Real spread across neighborhoods, from 21.0 up to 91.0, over 4 times the difference. Testing how well the neighborhood median would have predicted each house's actual known frontage, using `.transform()` to map each row to its own neighborhood's median without collapsing the DataFrame.

In [42]:
df["LotFrontage"].corr(df.groupby("Neighborhood")["LotFrontage"].transform("median"))

np.float64(0.4394131724301951)

0.439, slightly better than using `LotArea` alone (0.42), and more informed than a flat overall value. **Verdict:** filling missing `LotFrontage` using each house's own neighborhood median.

In [43]:
df["LotFrontage"] = df["LotFrontage"].fillna(df.groupby("Neighborhood")["LotFrontage"].transform("median"))

## Checkpoint

All missing values are resolved. Each fill was based on an actual investigation into why the value was missing, not a blanket mean, mode, or `"None"` applied without checking. Several columns needed individual correction for genuine data-entry gaps hiding inside otherwise clean "does not apply" patterns (Basement, MasVnr, Misc), and `LotFrontage` was filled using a validated neighborhood-based median rather than a flat value.

Saving to `train_cleaned.csv`. One thing worth noting for the next notebook: several columns were filled with the literal string `"None"`, which `pd.read_csv()` treats as a missing-value indicator by default. Reading this file back requires `keep_default_na=False, na_values=[]` to avoid reintroducing those values as NaN.

In [44]:
assert df.isnull().sum().sum() == 0

df.to_csv("../data/train_cleaned.csv", index=False)

check = pd.read_csv("../data/train_cleaned.csv", keep_default_na=False, na_values=[])
check.isnull().sum().sum()

np.int64(0)